# Hewwo Welcome to Fake or Real Review Project

# Step 1: Look at the big picture

# Step 2: Get the Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedShuffleSplit, StratifiedKFold, GridSearchCV
from sklearn import svm
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectFromModel
from sklearn.inspection import permutation_importance
import joblib
import random

In [ ]:
reviews_df = pd.read_csv('fake reviews dataset.csv')

In [ ]:
reviews_df.head()

,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...


In [ ]:
reviews_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40432 entries, 0 to 40431
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   category  40432 non-null  object 
 1   rating    40432 non-null  float64
 2   label     40432 non-null  object 
 3   text_     40432 non-null  object 
dtypes: float64(1), object(3)
memory usage: 1.2+ MB


In [ ]:
reviews_df.value_counts('label')

,count
label,
CG,20216
OR,20216


In [ ]:
reviews_df.describe(include="all")


,category,rating,label,text_
count,40432,40432.000000,40432,40432
unique,10,NaN,2,40412
top,Kindle_Store_5,NaN,CG,My dog loves these and it has kept her occupie...
freq,4730,NaN,20216,2
mean,NaN,4.256579,NaN,NaN
std,NaN,1.144354,NaN,NaN
min,NaN,1.000000,NaN,NaN
25%,NaN,4.000000,NaN,NaN
50%,NaN,5.000000,NaN,NaN
75%,NaN,5.000000,NaN,NaN


In [ ]:
reviews_df.value_counts("rating")

,count
rating,
5.0,24559
4.0,7965
3.0,3786
1.0,2155
2.0,1967


# clean the data

In [ ]:
# drop rows that have the text missing
reviews_df.dropna(subset=['text_'], inplace=True)

#convert float64 ratings to integers
reviews_df["rating"] = reviews_df["rating"].astype(int)

In [ ]:
from sklearn.preprocessing import LabelEncoder

# reviews_df.dropna(how="any") use this if eliminate any row that contains an usable value. i.e.
# This is commented out because we want to keep rows that have feature other than text missing. To handle these values we will
# use imputer rather than just deleting entire row.

# convert the labels of Computer Generated and Original to 0 and 1.

label_encoder = LabelEncoder()
label_encoder.fit(["CG","OR"])
reviews_df['label'] = label_encoder.transform(reviews_df['label'])

reviews_df.head()



,category,rating,label,text_
0,Home_and_Kitchen_5,5,0,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5,0,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5,0,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1,0,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5,0,Very nice set. Good quality. We have had the s...


# Split data into training and test data

In [ ]:
# creates a split object, that represents 1 split, with 20% of the data being used for testing and 80 for training.
# random_state could be any number, it just ensures that the split is reproducible,
# meaning that if you run the code multiple times, you will get the same split each time.
reviews_split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# reviews_split makes sure there is similar or same percentage of training and test sets.
# i.e. training set wont have 90% CG labels while testing set only have 10% CG labels.
# train_index and test_index are two NumPy arrays of row indices
# we get the data frame for both sets by passing it into the locate function of our original dataframe
for train_index, test_index in reviews_split.split(reviews_df, reviews_df["label"]):
  training_set = reviews_df.loc[train_index]
  testing_set = reviews_df.loc[test_index]

# x and y train are used for supervised learning. Labels shown to the model
x_train = training_set.drop(columns = ["label"])
y_train = training_set["label"]

# x and y test used for evaluated the trained model on supervised learning. Labels are not shown to the model.
x_test = testing_set.drop(columns = ['label'])
y_test = testing_set["label"]

display(x_train.info(),y_train.info())

<class 'pandas.core.frame.DataFrame'>
Index: 32345 entries, 25582 to 33847
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   category  32345 non-null  object
 1   rating    32345 non-null  int64 
 2   text_     32345 non-null  object
dtypes: int64(1), object(2)
memory usage: 1010.8+ KB
<class 'pandas.core.series.Series'>
Index: 32345 entries, 25582 to 33847
Series name: label
Non-Null Count  Dtype
--------------  -----
32345 non-null  int64
dtypes: int64(1)
memory usage: 505.4 KB


None

None

# Step 4: Prepare the Data for Machine Learning algorithms

## Data preprocessing pipelines (transformations)

In [ ]:
#Use one-hot encodign to conver the category strings in the category column to numerical values that the model can underestand
#Use scaling to scale rating numerical
#Use TF-IDF vectorizer to convert the review text into numerical features that the model can understand
#convert ratings to integers.


# note: might use small LLM to embed values for the text feature instead of TF-IDF to see which one gives more accurate results.

In [ ]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer

rating_feature = ['rating']
category_feature = ['category']
text_feature = 'text_'

#imputer handles the transformation of features that are either null or nan.
rating_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())

])
# one-hot converts column category into multiple columns with names of all values in the data. e.g. is_car, is_phone, is_radio instead of Category. Value in said rows
# are either 0 or 1 representing yes or no.
# unknown = "ignore" means that it will set all unknown values of new instances to 0s across the board for all the columns.
category_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value = "missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

text_pipeline = Pipeline([
  # not needed since we used dropna for all of invalid text values earlier
  # ("imputer", SimpleImputer(strategy="constant", fill_value="missing"))

  #tf-idf converts a string feature to a row of numbers. Each between 0-1 and represent each word.
  # i.e. Each new column name is each word, instead of a single column named text
  ('tf-idf', TfidfVectorizer(stop_words="english", max_features=5000))

])

preprocessing = ColumnTransformer([
    ("ratings", rating_pipeline, rating_feature),
    ("categories", category_pipeline, category_feature),
    ("text", text_pipeline, text_feature)
])

In [ ]:
# Manual implementaion of one-hot encoder. Column Transformer actually does this automatically. Can delete this block, but i kept it
# for reference

# one-hot encoding for category feature
from sklearn.preprocessing import OneHotEncoder
# set sparse to False to get a dense array(numpy array object) instead of a sparse matrix
# handle_uknown will ignore any categories in the test set that were not seen in the training set and will not raise an error when evaluating categories that have not been seen.
category_encoder = OneHotEncoder(sparse_output=False, handle_unknown = 'ignore')

# creates sparse matrix of one-hot encoded values for the category column
category_1hot_columns = category_encoder.fit_transform(reviews_df[['category']])
category_1hot_columns


# creates a new dataframe with the one-hot encoded values and column names base on the original category names
category_1hot_df = pd.DataFrame(category_1hot_columns, columns=category_encoder.get_feature_names_out())

# add the new columns to the original dataframe
reviews_df = pd.concat([reviews_df, category_1hot_df], axis =1)
reviews_df.head()
# delete the original "category"  column since we now have the one-hot encoded columns
reviews_df.drop(columns = ['category'])




In [ ]:
reviews_df.head()

,category,rating,label,text_
0,Home_and_Kitchen_5,5,0,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5,0,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5,0,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1,0,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5,0,Very nice set. Good quality. We have had the s...


# Step 5: Select a Model and Train it

## Baseline Model
we use the dummy model, because it is the minimum level intelligence that we need to beat to be considered useful



In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate


baseline_pipeline = Pipeline([
  ('prep', preprocessing),
  ('dummymodel', DummyClassifier(strategy="most_frequent"))
])

scoring = cross_validate(
  baseline_pipeline,
  x_train, y_train, scoring= ["accuracy", "precision", "recall", "average_precision"], cv = 5
)

scoring


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'fit_time': array([0.87593913, 0.80362558, 0.91521883, 1.14826655, 0.8681004 ]),
 'score_time': array([0.38501048, 0.39861703, 0.37396955, 0.68714285, 0.6796515 ]),
 'test_accuracy': array([0.49992271, 0.49992271, 0.49992271, 0.49992271, 0.49992271]),
 'test_precision': array([0.        , 0.        , 0.        , 0.49992271, 0.49992271]),
 'test_recall': array([0., 0., 0., 1., 1.]),
 'test_average_precision': array([0.50007729, 0.50007729, 0.50007729, 0.49992271, 0.49992271])}

## Inital Support Vector Machine

In [ ]:
from sklearn.svm import SVC

fake_review_classifier = make_pipeline(preprocessing, SVC())
fake_review_classifier.fit(x_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('ratings',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['rating']),
                                                 ('categories',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['category']),
                                                 ('text',
                                                  Pipeline(steps=[('tf-idf',
                                                                   TfidfVectorizer(max_features=5000,
                                                                                   stop_words='english'))]),
                                                  'text_')])),
                ('svc', SVC())])

## Cross validate to check scores of the initial SVM

In [ ]:
SVM_scoring = cross_validate(
  baseline_pipeline,
  x_train, y_train, scoring= ["accuracy", "precision", "recall", "average_precision"], cv = 5
)
print('''
===============
Printing Scores
===============\n
      ''', SVM_scoring
      )

print('''
===============
Printing mean of average_precision
===============\n
      ''',SVM_scoring["test_average_precision"].mean()
      )



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Printing Scores

       {'fit_time': array([0.80875826, 0.82054496, 0.80508614, 0.82892466, 1.23007035]), 'score_time': array([0.37285566, 0.40206194, 0.39864302, 0.57296824, 0.38894129]), 'test_accuracy': array([0.49992271, 0.49992271, 0.49992271, 0.49992271, 0.49992271]), 'test_precision': array([0.        , 0.        , 0.        , 0.49992271, 0.49992271]), 'test_recall': array([0., 0., 0., 1., 1.]), 'test_average_precision': array([0.50007729, 0.50007729, 0.50007729, 0.49992271, 0.49992271])}

Printing mean of average_precision

       0.5000154583397743


# Step 6: Fine-tune the Model

In [ ]:
param_grid = {
    "svc__C": [0.1, 1, 10],
    "svc__kernel": ["linear", "rbf"],
    "svc__gamma": ["scale", "auto"]
}

# Initialize the GridSearchCV (or RandomSearchCV if chosen) object
grid_search = GridSearchCV(estimator=fake_review_classifier, param_grid=param_grid, cv=5, scoring='average_precision', n_jobs=-1, verbose=3)

# Fit the grid(random) search to the data
grid_search.fit(x_train, y_train)

# Print the best parameters and the best score achieved during the grid search
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best parameters: {'svc__C': 10, 'svc__gamma': 'scale', 'svc__kernel': 'rbf'}
Best cross-validation score: 0.9358440007115021


# PCA Pipeline to visualize model

In [35]:

data_visualization_pipeline = Pipeline([
    ('preprocessing', preprocessing), # This is the object that produced the table above
    ('scaler', StandardScaler(with_mean=False)),
    ('pca', PCA(n_components=2))
])

# Perform PCA
x_train_pca = data_visualization_pipeline.fit_transform(x_train)

# Visualize weight of features in a dimensional space of 2.

In [36]:
# Pull out the PCA obeject.

pca_step = data_visualization_pipeline.named_steps['pca']

# GET NEW COLUMN NAMES
# Get new features that were made with preprocessing column transformer.
feature_names = data_visualization_pipeline.named_steps['preprocessing'].get_feature_names_out()


# create the loadings DataFrame : tells you exactly how much each of the original features (like Rating or the word "Excellent") contributed to the two new PCA axes.
# PC1 and PC2 because we chose n_components = 2, dimensional space of 2. PC1 = x axis and PC2 = y-axis. Each data point represents an instance i.e. (PC1,PC2)
# a high number in PC1 let, for example rating lets say has .90, means that rating has a strong influence on PC1(x-axis) to move in the right direction on the graph.
loadings_df = pd.DataFrame(
    pca_step.components_,
    columns=feature_names,
    index=['PC1', 'PC2']
)
# 4. Instead of just display(loadings_df), show the TOP 10 features because our text column is broken down into many many columns we cant just print it all out.
print("--- Top 10 Features for PC1 (X-axis) ---")
print(loadings_df.T['PC1'].sort_values(ascending=False).head(10))

print("\n--- Top 10 Features for PC2 (Y-axis) ---")
print(loadings_df.T['PC2'].sort_values(ascending=False).head(10))

display(loadings_df)

--- Top 10 Features for PC1 (X-axis) ---
text__ssl       0.273883
text__na        0.273701
text__url       0.268643
text__mp4       0.265180
text__images    0.263409
text__https     0.262267
text__png       0.254864
text__img       0.247860
text__slate     0.234746
text__div       0.232508
Name: PC1, dtype: float64

--- Top 10 Features for PC2 (Y-axis) ---
text__book                             0.196367
categories__category_Kindle_Store_5    0.191745
text__story                            0.172134
text__characters                       0.164054
text__read                             0.158459
categories__category_Books_5           0.149905
text__author                           0.124322
text__series                           0.114623
text__enjoyed                          0.113397
text__written                          0.105833
Name: PC2, dtype: float64


,ratings__rating,categories__category_Books_5,categories__category_Clothing_Shoes_and_Jewelry_5,categories__category_Electronics_5,categories__category_Home_and_Kitchen_5,categories__category_Kindle_Store_5,categories__category_Movies_and_TV_5,categories__category_Pet_Supplies_5,categories__category_Sports_and_Outdoors_5,categories__category_Tools_and_Home_Improvement_5,...,text__yrs,text__zero,text__zip,text__zipper,text__zippers,text__zoe,text__zombie,text__zombies,text__zone,text__zoom
PC1,0.000646,-0.012560,-0.002858,0.009490,0.005631,-0.016802,-0.00640,0.006267,0.006499,0.005482,...,-0.000271,-0.000225,0.000226,0.000356,-0.00003,-0.000410,-0.000653,-0.000713,-0.000222,0.000219
PC2,0.004477,0.149905,-0.073408,-0.064703,-0.063217,0.191745,0.07388,-0.060586,-0.064859,-0.070122,...,-0.001737,-0.002394,-0.007247,-0.015472,-0.00660,0.006415,0.007051,0.007462,0.007911,-0.004576


# Plot 2D representation of Model

# Step 7: Present the Solution

# Step 8: Launch, Monitor, and Maintain the System

In [ ]:
#first Code block y